# Flow Matching

在[理论部分](./theory.md)，我们详细讲解了 Flow Matching 是如何实现，这里我们就依据理论部分的讲解，采用 **OT 路径** 实现一个简单的 Flow Matching 的示例

<div align="center">
    <img src="./imgs/FM_examples.png" alt="FM_examples" style="width: 600px; height: auto;">
</div>

本篇notebook基于如下链接的内容和代码进行整理，参考链接：

[参考知乎解析](https://zhuanlan.zhihu.com/p/1946706315792619506)

[FM 论文](https://arxiv.org/pdf/2210.02747)

[FM Pytorch 参考实现](https://github.com/facebookresearch/flow_matching)

## 训练

和 DDPM 一致，我们的模型依旧使用 **UNet** 来完成，这次我们初始化 UNet 就使用小一些的参数，只针对一张图进行训练和采样，展示一下 Flow Matching 的过程

那么我们的训练过程如下:

1. 从均匀分布中采样出一批时间步$t$，从目标分布$q(x)$中采样终点$x_1$，从先验分布$p(x)$采样起点$x_0$，三者数量相同，均为 batch_size

2. 根据 OT 路径的公式$x_t = (1 - t)x_0 + tx_1$计算出$t$时刻的对应位置$x_t$

3. 输入($x_t, t$)到模型中，预测当前时刻的速度场$v_t(x_t)$

4. 计算与参考速度场$u_t(x_t) = x_1 - x_0$的均方误差，并反向传播更新模型参数

5. 重复直至收敛或训练结束

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from c_fm import get_config, get_unet
import os
from tqdm import tqdm
from torchvision import transforms, datasets
from torch.utils.data import DataLoader


# -----------------------------------------------------------------------------------
# 训练超参数
# -----------------------------------------------------------------------------------
BATCH_SIZE = 1
IMG_SIZE = 64  # 和config的global里的img_size一致
EPOCHS = 10000
LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_SAVE_DIR = "./ckpts/"
TRAIN_DIR = "./train"


In [ ]:
# -----------------------------------------------------------------------------------
# 数据加载与预处理
# -----------------------------------------------------------------------------------
os.makedirs(CHECKPOINT_SAVE_DIR, exist_ok=True)

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)


In [6]:
# -----------------------------------------------------------------------------------
# OT Path 路径获取
# -----------------------------------------------------------------------------------
def get_ot_path(x0:torch.Tensor, x1:torch.Tensor, t:torch.Tensor):
    # x0/x1: [bs, c, h, w] | t: [bs]
    # t变为[bs, 1, 1, 1]，运算时就会广播成[bs, c, h, w]从而逐元素加减
    t = t[:, None, None, None]

    xt = (1 - t)*x0 + t*x1

    return xt 


x0 = torch.randn(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE)
x1 = torch.randn(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE)
t = torch.rand(BATCH_SIZE)
print(f'x0/x1 size: {x0.size()}')
print(f"t size: {t.size()}")
xt = get_ot_path(x0, x1, t)
print(f"Output xt size: {xt.size()}")


x0/x1 size: torch.Size([1, 3, 64, 64])
t size: torch.Size([1])
Output xt size: torch.Size([1, 3, 64, 64])


In [9]:
# -----------------------------------------------------------------------------------
# 训练，注意参考速度场ut永远是(x1-x0)
# -----------------------------------------------------------------------------------
pbar = tqdm(range(EPOCHS), desc="Training")
config = get_config("./configs/fm_unet.yaml")
unet = get_unet(config)
unet.train()
optimizer = optim.AdamW(unet.parameters(), lr=LR)

for epoch in pbar:
    total_loss = 0.0

    for imgs, _ in dataloader:
        x1 = imgs.to(DEVICE)
        x0 = torch.randn_like(x1).to(DEVICE)
        t = torch.rand(x0.shape[0]).to(DEVICE)

        xt = get_ot_path(x0, x1, t).to(DEVICE)
        ut = x1 - x0
        vt = unet(xt, t)

        optimizer.zero_grad()
        loss = torch.mean((vt - ut) ** 2)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() 

    avg_loss = total_loss / len(dataloader)
    pbar.set_postfix({"avg_loss": round(avg_loss, 6)})

    if (epoch + 1) % 2000 == 0:
        torch.save(
            unet.state_dict(),
            os.path.join(CHECKPOINT_SAVE_DIR, f"fm_unet_epoch_{epoch+1}.pth")
        )
        print(f'[trainer]Saved: fm_unet_epoch_{epoch+1}.pth')

torch.save(
    unet.state_dict(),
    os.path.join(CHECKPOINT_SAVE_DIR, f"fm_unet_iter{EPOCHS}.pth")
)
print(f'[trainer]Saved: fm_unet_iter{EPOCHS}.pth')


Training:  20%|██        | 2003/10000 [01:32<06:23, 20.87it/s, avg_loss=0.0383]

[trainer]Saved: fm_unet_epoch_2000.pth


Training:  40%|████      | 4001/10000 [03:03<04:45, 20.98it/s, avg_loss=0.0114]

[trainer]Saved: fm_unet_epoch_4000.pth


Training:  60%|██████    | 6003/10000 [04:35<03:14, 20.56it/s, avg_loss=0.00844]

[trainer]Saved: fm_unet_epoch_6000.pth


Training:  80%|████████  | 8003/10000 [06:09<01:35, 21.01it/s, avg_loss=0.00695]

[trainer]Saved: fm_unet_epoch_8000.pth


Training: 100%|██████████| 10000/10000 [07:41<00:00, 21.65it/s, avg_loss=0.0125]

[trainer]Saved: fm_unet_epoch_10000.pth
[trainer]Saved: fm_unet_iter10000.pth


## 采样

采样的过程如下:

1. 从标准高斯分布中采样张噪声图$x_0$

2. 循环num_steps步，num_steps可以自定义，记为$n$:
    - 将当前的$(x_t, t)$输入到模型中，预测出$v_t(x_t)$

    - 计算下一个时间步的位置: $x_{t+1} = x_t + v_t(x_t) \cdot \frac{1}{n}$

3. 最终的$x_1$即为采样图像

In [3]:
from torchvision.utils import save_image


def sample(num_samples, num_steps, unet, save_dir="./samples"):
    os.makedirs(save_dir, exist_ok=True)
    
    x0 = torch.randn(num_samples, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    dt = 1.0 / num_steps
    
    xt = x0
    with torch.no_grad():
        for i in tqdm(range(num_steps), desc="Sampling"):
            t = torch.full((num_samples,), i * dt, device=DEVICE)
            vt = unet(xt, t)

            xt = xt + vt*dt

    x1 = xt

    # range(-1, 1) -> range(0, 1)
    x1 = (x1 + 1) / 2.0
    x1 = torch.clamp(x1, 0.0, 1.0)

    for i in range(num_samples):
        save_image(x1[i], os.path.join(save_dir, f"sample{i}.png"))

    print(f'Saved {num_samples} images at: {save_dir}')


In [8]:
from c_fm import get_config, get_unet
import os
import torch 


config = get_config("./configs/fm_unet.yaml")
unet_pretrained = get_unet(config, load_pretrained=True)
unet_pretrained.eval()
sample(4, 1000, unet_pretrained)


Loaded checkpoint from ./ckpts/fm_unet_iter10000.pth


Sampling: 100%|██████████| 1000/1000 [00:14<00:00, 70.39it/s]


Saved 4 images at: ./samples


结果如下:

<div align="center">
    <img src="./samples/sample0.png" style="width: 100px; height: auto;">
    <img src="./samples/sample1.png" style="width: 100px; height: auto;">
    <img src="./samples/sample2.png" style="width: 100px; height: auto;">
    <img src="./samples/sample3.png" style="width: 100px; height: auto;">
</div>


我们可以可视化每一个时间步的采样过程

In [9]:
from c_fm import visualize_sampling


visualize_sampling(unet_pretrained, num_steps=1000, img_size=IMG_SIZE, 
                   frame_interval=50, gif_duration=80)


Sampling for visualization: 100%|██████████| 1000/1000 [00:11<00:00, 86.82it/s]


GIF 如下:

<div align="center">
    <img src="./visualize/sampling_process.gif" alt="sampling_process.gif" style="width: 100px; height: auto;">
</div>

过程图如下:

<div align="center">
    <img src="./visualize/sampling_process.png" alt="sampling_process.png" style="width: 600px; height: auto;">
</div>
